### **Import Library**

In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.append('.')  
from data import supply, demand, cost, agents, destinations

### **Cost Matrix**

In [2]:
short_destinations = [d.split('(')[-1].replace(')', '') for d in destinations]
short_agents = ['A1', 'A2', 'A3']

df_cost = pd.DataFrame(
    cost,
    index=short_agents,
    columns=short_destinations
)

df_cost.index.name = 'Dari/Ke'
df_cost['Supply'] = supply
demand_row = demand + [sum(supply)]
df_cost.loc['Demand'] = demand_row

df_cost


,CS,KD,BN,PG,KT,CB,CSK,PB,CH,CJ,Supply
Dari/Ke,,,,,,,,,,,
A1,39.1,40.2,37.6,40.6,43.1,40.0,41.0,40.6,41.6,37.6,600
A2,43.7,44.1,45.1,42.1,51.8,42.6,41.1,43.1,40.1,42.7,450
A3,42.1,40.6,43.1,44.5,43.3,45.3,46.3,41.4,42.3,45.8,250
Demand,300.0,100.0,200.0,300.0,50.0,100.0,50.0,50.0,100.0,50.0,1300


### **Algoritma Least Cost Method**

In [3]:
def least_cost_method(supply, demand, cost):
    m = len(supply)     
    n = len(demand)       
    
    s = supply.copy()
    d = demand.copy()
    c = [row.copy() for row in cost]
    
    allocation = np.zeros((m, n)) 
    steps = []
    step_num = 1
    
    while True:
        min_cost = float('inf')
        min_i, min_j = -1, -1
        
        for i in range(m):
            for j in range(n):
                if c[i][j] < min_cost and s[i] > 0 and d[j] > 0:
                    min_cost = c[i][j]
                    min_i, min_j = i, j
        
        if min_i == -1:
            break
        
        qty = min(s[min_i], d[min_j])
        allocation[min_i][min_j] = qty
        s[min_i] -= qty
        d[min_j] -= qty
        
        steps.append({
            'Langkah': step_num,
            'Dari'   : agents[min_i].split('(')[0].strip(),
            'Ke'     : destinations[min_j],
            'Biaya'  : min_cost,
            'Alokasi': qty,
            'Sisa Supply': s[min_i],
            'Sisa Demand': d[min_j]
        })
        step_num += 1
        
        if s[min_i] == 0:
            for j in range(n):
                c[min_i][j] = float('inf')
        if d[min_j] == 0:
            for i in range(m):
                c[i][min_j] = float('inf')
    
    return allocation, steps


In [4]:
allocation, steps = least_cost_method(supply, demand, cost)
df_steps = pd.DataFrame(steps)


In [5]:
df_steps

,Langkah,Dari,Ke,Biaya,Alokasi,Sisa Supply,Sisa Demand
0,1,Agen 1,Bojong Nangka (BN),37.6,200,400,0
1,2,Agen 1,Cijantra (CJ),37.6,50,350,0
2,3,Agen 1,Curug Sangereng (CS),39.1,300,50,0
3,4,Agen 1,Cibogo (CB),40.0,50,0,50
4,5,Agen 2,Cihuni (CH),40.1,100,350,0
5,6,Agen 3,Kelapa Dua (KD),40.6,100,150,0
6,7,Agen 2,Cisauk (CSK),41.1,50,300,0
7,8,Agen 3,Pakulonan Barat (PB),41.4,50,100,0
8,9,Agen 2,Pagedangan (PG),42.1,300,0,0
9,10,Agen 3,Karang Tengah (KT),43.3,50,50,0


In [6]:
df_alloc = pd.DataFrame(
    allocation,
    index=['A1', 'A2', 'A3'],
    columns=short_destinations
)
df_alloc.index.name = 'Dari/Ke'

df_alloc['Supply'] = supply
df_alloc.loc['Demand'] = demand + [sum(supply)]

def fmt(val):
    return '-' if val == 0 else int(val)

df_alloc_display = df_alloc.copy()
df_alloc_display = df_alloc_display.map(
    lambda x: '-' if x == 0.0 else (int(x) if x == int(x) else x)
)

In [7]:
df_alloc_display

,CS,KD,BN,PG,KT,CB,CSK,PB,CH,CJ,Supply
Dari/Ke,,,,,,,,,,,
A1,300,-,200,-,-,50,-,-,-,50,600
A2,-,-,-,300,-,-,50,-,100,-,450
A3,-,100,-,-,50,50,-,50,-,-,250
Demand,300,100,200,300,50,100,50,50,100,50,1300


### **Perhitungan Total Biaya Awal (Z)**

In [8]:
cost_np = np.array(cost)
total_cost = np.sum(allocation * cost_np)

In [9]:
rincian = []
for i in range(len(supply)):
    for j in range(len(demand)):
        if allocation[i][j] > 0:
            subtotal = allocation[i][j] * cost[i][j]
            rincian.append({
                'Dari'    : agents[i].split('(')[0].strip(),
                'Ke'      : destinations[j],
                'Alokasi' : int(allocation[i][j]),
                'Biaya/unit (Rp. rb)': cost[i][j],
                'Subtotal (Rp. rb)': subtotal
            })


In [10]:
df_rincian = pd.DataFrame(rincian)
print(df_rincian.to_string(index=False))

  Dari                   Ke  Alokasi  Biaya/unit (Rp. rb)  Subtotal (Rp. rb)
Agen 1 Curug Sangereng (CS)      300                 39.1            11730.0
Agen 1   Bojong Nangka (BN)      200                 37.6             7520.0
Agen 1          Cibogo (CB)       50                 40.0             2000.0
Agen 1        Cijantra (CJ)       50                 37.6             1880.0
Agen 2      Pagedangan (PG)      300                 42.1            12630.0
Agen 2         Cisauk (CSK)       50                 41.1             2055.0
Agen 2          Cihuni (CH)      100                 40.1             4010.0
Agen 3      Kelapa Dua (KD)      100                 40.6             4060.0
Agen 3   Karang Tengah (KT)       50                 43.3             2165.0
Agen 3          Cibogo (CB)       50                 45.3             2265.0
Agen 3 Pakulonan Barat (PB)       50                 41.4             2070.0


In [11]:
print(f"\nRumus Z:")
rumus_parts = [f"({int(r['Alokasi'])} × {r['Biaya/unit (Rp. rb)']})" for r in rincian]
print("Z = " + " + ".join(rumus_parts))
print(f"\nZ = {' + '.join([str(r['Subtotal (Rp. rb)']) for r in rincian])}")
print(f"  Total Biaya Awal (Z) = Rp. {total_cost:,.1f} ribu")
print(f"                       = Rp. {total_cost * 1000:,.0f}")


Rumus Z:
Z = (300 × 39.1) + (200 × 37.6) + (50 × 40.0) + (50 × 37.6) + (300 × 42.1) + (50 × 41.1) + (100 × 40.1) + (100 × 40.6) + (50 × 43.3) + (50 × 45.3) + (50 × 41.4)

Z = 11730.0 + 7520.0 + 2000.0 + 1880.0 + 12630.0 + 2055.0 + 4010.0 + 4060.0 + 2165.0 + 2265.0 + 2070.0
  Total Biaya Awal (Z) = Rp. 52,385.0 ribu
                       = Rp. 52,385,000


### **Load Hasil**

In [12]:
np.save('allocation_lc.npy', allocation)